# Sudoku-07 : Résolution par Propagation de Contraintes (Norvig) (Rust)

**Niveau** : Programmation par contraintes | **Durée** : courte | **Prérequis** : Sudoku-06 AIMA-CSP (Python)

## Navigation

| << Précédent | Sommaire | Suivant >> |
|--------------|----------|------------|
| [Sudoku-06-AIMA-CSP-Python](Sudoku-06-AIMA-CSP-Python.ipynb) | [Index](README.md) | [Sudoku-08-HumanStrategies-Python](Sudoku-08-HumanStrategies-Python.ipynb) |

## Objectifs d'apprentissage

- Comprendre l'approche élégante de Peter Norvig pour résoudre le Sudoku
- Implémenter un solveur basé sur la propagation de contraintes et le backtracking
- Combiner l'élimination des candidats (pairs et unités) avec une recherche en profondeur guidée par l'heuristique MRV
- Comparer les performances sur différents niveaux de difficulté

## Configuration et Imports

In [2]:
// Imports
// std seul : HashMap, HashSet, Instant, io — aucun crate externe
println!("Solveur Sudoku - Approche Norvig");

Solveur Sudoku - Approche Norvig


## Introduction : L'approche Norvig

Peter Norvig a proposé en 2006 une solution élégante et compacte en Python. L'idée clé :

1. **Représentation** : Chaque case contient l'ensemble des valeurs possibles
2. **Propagation** : Éliminer les valeurs impossibles (Constraint Propagation)
3. **Backtracking** : Si la propagation bloque, choisir une case et essayer une valeur

### Heuristiques clés

- **MRV (Minimum Remaining Values)** : Choisir la case avec le moins de valeurs possibles
- **Forward Checking** : Propager immédiatement les conséquences de chaque choix

> **Lien** : [Solving Every Sudoku Puzzle](https://norvig.com/sudoku.html) par Peter Norvig

## Définition des constantes

Nous définissons la structure du Sudoku : 81 cases, 27 unités (rangées, colonnes et blocs).

In [3]:
// Définition des constantes
use std::collections::{HashMap, HashSet};

let digits = "123456789";
let rows = "ABCDEFGHI";
let cols = digits;

let squares: Vec<String> = rows.chars()
    .flat_map(|r| cols.chars().map(move |c| format!("{}{}", r, c)))
    .collect();

// Unites : colonnes, lignes, blocs (ordre de la source)
let mut unitlist: Vec<Vec<String>> = Vec::new();
for c in cols.chars() {
    unitlist.push(rows.chars().map(|r| format!("{}{}", r, c)).collect());
}
for r in rows.chars() {
    unitlist.push(cols.chars().map(|c| format!("{}{}", r, c)).collect());
}
for rs in ["ABC", "DEF", "GHI"] {
    for cs in ["123", "456", "789"] {
        unitlist.push(rs.chars()
            .flat_map(|r| cs.chars().map(move |c| format!("{}{}", r, c)))
            .collect());
    }
}

// Dictionnaire : square -> liste des 3 unités
let units: HashMap<String, Vec<Vec<String>>> = squares.iter()
    .map(|s| (s.clone(), unitlist.iter().filter(|u| u.contains(s)).cloned().collect()))
    .collect();

// Dictionnaire : square -> ensemble des 20 pairs
let peers: HashMap<String, HashSet<String>> = squares.iter()
    .map(|s| {
        let mut p: HashSet<String> = HashSet::new();
        for u in &units[s] {
            for sq in u { p.insert(sq.clone()); }
        }
        p.remove(s);
        (s.clone(), p)
    })
    .collect();

// Formatage Python (apostrophes, deux niveaux)
fn py_list(items: &[String]) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}
fn py_nested(items: &[Vec<String>]) -> String {
    format!("[{}]", items.iter().map(|inner| py_list(inner)).collect::<Vec<_>>().join(", "))
}

println!("Carres: {}", squares.len());
println!("Unites: {}", unitlist.len());
println!("Pairs de A1: {}", peers["A1"].len());
println!("Unites de A1: {}", py_nested(&units["A1"]));

Carres: 81


Unites: 27


Pairs de A1: 20


Unites de A1: [['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1', 'I1'], ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9'], ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3']]


### Lecture des constantes : 81, 27, 20 — l'arithmétique du voisinage

La sortie donne les trois nombres qui structurent tout l'algorithme. **81 carrés** : les cellules A1-I9. **27 unités** : une par ligne, colonne et bloc — la contrainte du jeu dit qu'une unité est une permutation de 1-9. **20 pairs de A1** : le voisinage d'une cellule — lisez la décomposition : 8 camarades de colonne + 8 camarades de ligne + 4 camarades de bloc restants = **20**, et les trois unités de A1 sont listées explicitement (colonne, ligne, bloc 3x3). C'est ce voisinage que `eliminate` parcourt : quand une valeur est fixée dans une cellule, ses 20 pairs la perdent immédiatement. Toute la puissance de l'approche Norvig tient dans cette table précalculée — construite une fois, interrogée des millions de fois.


Un dernier détail de lecture : les trois unités de A1 sont listées **colonne d'abord** — c'est l'ordre d'itération du dictionnaire des unités par carré, aucun ordre n'étant sémantiquement privilégié pour l'algorithme ; seul compte le contenu des 27 ensembles.

## Propagation des contraintes

Les deux opérations fondamentales :

1. **assign(values, square, digit)** : Assigne un chiffre à une case
   - Élimine ce chiffre de tous les pairs de la case

2. **eliminate(values, square, digit)** : Élimine un chiffre possible d'une case
   - Si une case n'a plus qu'une valeur, l'éliminer des pairs
   - Si une unité n'a qu'une place pour un chiffre, l'assigner

### Règles de propagation

| Règle | Description |
|-------|-------------|
| **Peer Elimination** | Si A1 = 1, alors 1 est éliminé de tous les pairs de A1 |
| **Singleton Unit** | Si seule la case A1 peut contenir 1 dans son bloc, alors A1 = 1 |

### Anatomie d'assign/eliminate : l'inversion qui fait tout travailler

Le cœur de Norvig tient en une inversion : `assign(values, s, d)` n'écrit pas « la cellule s vaut d » — il appelle `eliminate` pour retirer **toutes les autres valeurs** de s. Et `eliminate(values, s, d)` fait le vrai travail, en trois temps : (a) retirer d des candidats de s ; (b) si s est réduit à un seul candidat, **éliminer cette valeur de tous ses 20 pairs** (récursion en cascade) ; (c) si une unité de s ne peut plus placer d que dans une seule cellule, **y assigner d** (le singleton caché). Chaque elimination déclenche donc potentiellement d'autres assignations — c'est cette propagation mutuellement récursive qui résout l'essentiel des grilles avant même que `search` ne branche.


Et le lien avec `search` est d'une élégance minimaliste : MRV (choisir la cellule au domaine minimal) s'y écrit en Rust `squares.iter().filter(|s| values[s].len() > 1).min_by_key(|s| (values[s].len(), s.clone()))` — le domaine **étant** la chaîne de candidats, l'heuristique tient en un tri par longueur de chaîne, avec départage lexicographique sur le nom de case en cas d'égalité.

In [4]:
use std::collections::{HashMap, HashSet};

#[derive(Clone)]
struct SudokuData {
    squares: Vec<String>,
    units: HashMap<String, Vec<Vec<String>>>,
    peers: HashMap<String, HashSet<String>>,
}

let digits = "123456789";
let rows = "ABCDEFGHI";
let cols = digits;

let squares: Vec<String> = rows.chars()
    .flat_map(|r| cols.chars().map(move |c| format!("{}{}", r, c)))
    .collect();

let mut unitlist: Vec<Vec<String>> = Vec::new();
for c in cols.chars() {
    unitlist.push(rows.chars().map(|r| format!("{}{}", r, c)).collect());
}
for r in rows.chars() {
    unitlist.push(cols.chars().map(|c| format!("{}{}", r, c)).collect());
}
for rs in ["ABC", "DEF", "GHI"] {
    for cs in ["123", "456", "789"] {
        unitlist.push(rs.chars()
            .flat_map(|r| cs.chars().map(move |c| format!("{}{}", r, c)))
            .collect());
    }
}

let units: HashMap<String, Vec<Vec<String>>> = squares.iter()
    .map(|s| (s.clone(), unitlist.iter().filter(|u| u.contains(s)).cloned().collect()))
    .collect();

let peers: HashMap<String, HashSet<String>> = squares.iter()
    .map(|s| {
        let mut p: HashSet<String> = HashSet::new();
        for u in &units[s] { for sq in u { p.insert(sq.clone()); } }
        p.remove(s);
        (s.clone(), p)
    })
    .collect();

let data = SudokuData { squares, units, peers };

// Assigne d à s en éliminant toutes les autres valeurs.
// Renvoie true si succès, false si contradiction.
fn assign(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    let other_values: String = values[s].chars().filter(|&c| c != d).collect();
    for d2 in other_values.chars() {
        if !eliminate(data, values, s, d2) {
            return false;
        }
    }
    true
}

// Élimine d de values[s] ; propage les conséquences.
fn eliminate(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    if !values[s].contains(d) {
        // Déjà éliminé
        return true;
    }
    let new_val: String = values[s].chars().filter(|&c| c != d).collect();
    values.insert(s.to_string(), new_val);

    let s_val = values[s].clone();
    // (1) Si une case n'a qu'une valeur, l'éliminer des pairs
    if s_val.is_empty() {
        // Contradiction: plus de valeurs
        return false;
    } else if s_val.len() == 1 {
        let d2 = s_val.chars().next().unwrap();
        for s2 in &data.peers[s] {
            if !eliminate(data, values, s2, d2) {
                return false;
            }
        }
    }

    // (2) Si une unité n'a qu'une place pour d, l'y assigner
    for u in &data.units[s] {
        let dplaces: Vec<&String> = u.iter().filter(|s2| values[s2.as_str()].contains(d)).collect();
        if dplaces.is_empty() {
            // Contradiction: pas de place pour d
            return false;
        } else if dplaces.len() == 1 {
            if !assign(data, values, &dplaces[0], d) {
                return false;
            }
        }
    }
    true
}

println!("Fonctions definies : assign, eliminate");

Fonctions definies : assign, eliminate


## Parsing d'une grille

Une grille est représentée comme un dictionnaire `{square: digits}` où `digits` est :
- Un chiffre ('1'-'9') si la case est résolue
- Une chaîne de chiffres possibles si la case n'est pas résolue

Format d'entrée : chaîne de 81 caractères (ou avec `.` et `0` pour les cases vides).

In [5]:
use std::collections::{HashMap, HashSet};

#[derive(Clone)]
struct SudokuData {
    squares: Vec<String>,
    units: HashMap<String, Vec<Vec<String>>>,
    peers: HashMap<String, HashSet<String>>,
}

let digits = "123456789";
let rows = "ABCDEFGHI";
let cols = digits;

let squares: Vec<String> = rows.chars()
    .flat_map(|r| cols.chars().map(move |c| format!("{}{}", r, c)))
    .collect();

let mut unitlist: Vec<Vec<String>> = Vec::new();
for c in cols.chars() {
    unitlist.push(rows.chars().map(|r| format!("{}{}", r, c)).collect());
}
for r in rows.chars() {
    unitlist.push(cols.chars().map(|c| format!("{}{}", r, c)).collect());
}
for rs in ["ABC", "DEF", "GHI"] {
    for cs in ["123", "456", "789"] {
        unitlist.push(rs.chars()
            .flat_map(|r| cs.chars().map(move |c| format!("{}{}", r, c)))
            .collect());
    }
}

let units: HashMap<String, Vec<Vec<String>>> = squares.iter()
    .map(|s| (s.clone(), unitlist.iter().filter(|u| u.contains(s)).cloned().collect()))
    .collect();

let peers: HashMap<String, HashSet<String>> = squares.iter()
    .map(|s| {
        let mut p: HashSet<String> = HashSet::new();
        for u in &units[s] { for sq in u { p.insert(sq.clone()); } }
        p.remove(s);
        (s.clone(), p)
    })
    .collect();

let data = SudokuData { squares, units, peers };

fn assign(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    let other_values: String = values[s].chars().filter(|&c| c != d).collect();
    for d2 in other_values.chars() {
        if !eliminate(data, values, s, d2) { return false; }
    }
    true
}

fn eliminate(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    if !values[s].contains(d) { return true; }
    let new_val: String = values[s].chars().filter(|&c| c != d).collect();
    values.insert(s.to_string(), new_val);
    let s_val = values[s].clone();
    if s_val.is_empty() { return false; }
    else if s_val.len() == 1 {
        let d2 = s_val.chars().next().unwrap();
        for s2 in &data.peers[s] {
            if !eliminate(data, values, s2, d2) { return false; }
        }
    }
    for u in &data.units[s] {
        let dplaces: Vec<&String> = u.iter().filter(|s2| values[s2.as_str()].contains(d)).collect();
        if dplaces.is_empty() { return false; }
        else if dplaces.len() == 1 {
            if !assign(data, values, &dplaces[0], d) { return false; }
        }
    }
    true
}

// Convertit une grille en dictionnaire {square: char}.
fn grid_values(grid: &str, squares: &[String]) -> Vec<(String, char)> {
    let chars: Vec<char> = grid.chars()
        .filter(|c| c.is_ascii_digit() || *c == '0' || *c == '.')
        .collect();
    assert_eq!(chars.len(), 81);
    squares.iter().zip(chars).map(|(s, c)| (s.clone(), c)).collect()
}

// Convertit une grille string en dictionnaire {square: digits}.
fn parse_grid(data: &SudokuData, grid: &str) -> Option<HashMap<String, String>> {
    let digits = "123456789";
    // Toutes les valeurs possibles
    let mut values: HashMap<String, String> = data.squares.iter()
        .map(|s| (s.clone(), digits.to_string()))
        .collect();
    for (s, d) in grid_values(grid, &data.squares) {
        if d != '0' && d != '.' {
            if !assign(data, &mut values, s.as_str(), d) {
                // Contradiction
                return None;
            }
        }
    }
    Some(values)
}

// Test
let test_grid = "..3.2.6..9..3.5..1..18.64....81.29..7.......8..67.82....26.95..8..2.3..9..5.1.3..";
println!("Grille test: {} caracteres", test_grid.len());

Grille test: 81 caracteres


### Lecture de l'encodage : 81 caractères, un par carré

La grille est une **chaîne de 81 caractères** lue ligne par ligne : un chiffre = un indice fixé, un point ou un zéro = une cellule libre. C'est l'encodage canonique de l'essai de Norvig — compact, insensible aux espaces, facile à copier-coller dans un test. Au parsing, chaque point devient le domaine complet `123456789` et chaque indice un domaine singleton : le solveur ne verra plus jamais la différence entre les deux — tout son travail consiste à réduire des ensembles de candidats jusqu'à ce que chacun soit un singleton.

## Résolution avec Backtracking

Si la propagation ne suffit pas, on utilise le backtracking avec l'heuristique MRV :

1. Choisir la case avec le **moins de valeurs possibles** (Minimum Remaining Values)
2. Essayer chaque valeur possible
3. Récursivement résoudre
4. Si échec, backtracker

### Pourquoi MRV ?

- Maximise les chances d'échec rapide (pruning)
- Réduit la profondeur de l'arbre de recherche
- Est souvent couplé avec LCV (Least Constraining Value)

In [6]:
use std::collections::{HashMap, HashSet};

#[derive(Clone)]
struct SudokuData {
    squares: Vec<String>,
    units: HashMap<String, Vec<Vec<String>>>,
    peers: HashMap<String, HashSet<String>>,
}

let digits = "123456789";
let rows = "ABCDEFGHI";
let cols = digits;

let squares: Vec<String> = rows.chars()
    .flat_map(|r| cols.chars().map(move |c| format!("{}{}", r, c)))
    .collect();

let mut unitlist: Vec<Vec<String>> = Vec::new();
for c in cols.chars() {
    unitlist.push(rows.chars().map(|r| format!("{}{}", r, c)).collect());
}
for r in rows.chars() {
    unitlist.push(cols.chars().map(|c| format!("{}{}", r, c)).collect());
}
for rs in ["ABC", "DEF", "GHI"] {
    for cs in ["123", "456", "789"] {
        unitlist.push(rs.chars()
            .flat_map(|r| cs.chars().map(move |c| format!("{}{}", r, c)))
            .collect());
    }
}

let units: HashMap<String, Vec<Vec<String>>> = squares.iter()
    .map(|s| (s.clone(), unitlist.iter().filter(|u| u.contains(s)).cloned().collect()))
    .collect();

let peers: HashMap<String, HashSet<String>> = squares.iter()
    .map(|s| {
        let mut p: HashSet<String> = HashSet::new();
        for u in &units[s] { for sq in u { p.insert(sq.clone()); } }
        p.remove(s);
        (s.clone(), p)
    })
    .collect();

let data = SudokuData { squares, units, peers };

fn assign(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    let other_values: String = values[s].chars().filter(|&c| c != d).collect();
    for d2 in other_values.chars() {
        if !eliminate(data, values, s, d2) { return false; }
    }
    true
}

fn eliminate(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    if !values[s].contains(d) { return true; }
    let new_val: String = values[s].chars().filter(|&c| c != d).collect();
    values.insert(s.to_string(), new_val);
    let s_val = values[s].clone();
    if s_val.is_empty() { return false; }
    else if s_val.len() == 1 {
        let d2 = s_val.chars().next().unwrap();
        for s2 in &data.peers[s] {
            if !eliminate(data, values, s2, d2) { return false; }
        }
    }
    for u in &data.units[s] {
        let dplaces: Vec<&String> = u.iter().filter(|s2| values[s2.as_str()].contains(d)).collect();
        if dplaces.is_empty() { return false; }
        else if dplaces.len() == 1 {
            if !assign(data, values, &dplaces[0], d) { return false; }
        }
    }
    true
}

fn grid_values(grid: &str, squares: &[String]) -> Vec<(String, char)> {
    let chars: Vec<char> = grid.chars()
        .filter(|c| c.is_ascii_digit() || *c == '0' || *c == '.')
        .collect();
    assert_eq!(chars.len(), 81);
    squares.iter().zip(chars).map(|(s, c)| (s.clone(), c)).collect()
}

fn parse_grid(data: &SudokuData, grid: &str) -> Option<HashMap<String, String>> {
    let digits = "123456789";
    let mut values: HashMap<String, String> = data.squares.iter()
        .map(|s| (s.clone(), digits.to_string()))
        .collect();
    for (s, d) in grid_values(grid, &data.squares) {
        if d != '0' && d != '.' {
            if !assign(data, &mut values, s.as_str(), d) { return None; }
        }
    }
    Some(values)
}

fn search(data: &SudokuData, values: Option<HashMap<String, String>>) -> Option<HashMap<String, String>> {
    let values = values?;
    if data.squares.iter().all(|s| values[s.as_str()].len() == 1) {
        return Some(values);
    }
    // MRV : min par (longueur, nom) en itérant sur squares (ordre lexicographique)
    let s = data.squares.iter()
        .filter(|s| values[s.as_str()].len() > 1)
        .min_by_key(|s| (values[s.as_str()].len(), s.to_string()))
        .unwrap();
    let candidates = values[s.as_str()].clone();
    for d in candidates.chars() {
        let mut new_values = values.clone();
        if assign(data, &mut new_values, s.as_str(), d) {
            if let Some(result) = search(data, Some(new_values)) {
                return Some(result);
            }
        }
    }
    None
}

fn solve(data: &SudokuData, grid: &str) -> Option<HashMap<String, String>> {
    search(data, parse_grid(data, grid))
}

println!("Fonctions definies : solve, search (backtracking MRV)");

Fonctions definies : solve, search (backtracking MRV)


## Affichage des résultats

La fonction `display()` formate une grille de manière lisible.

In [7]:
use std::collections::{HashMap, HashSet};

#[derive(Clone)]
struct SudokuData {
    squares: Vec<String>,
    units: HashMap<String, Vec<Vec<String>>>,
    peers: HashMap<String, HashSet<String>>,
}

let digits = "123456789";
let rows = "ABCDEFGHI";
let cols = digits;

let squares: Vec<String> = rows.chars()
    .flat_map(|r| cols.chars().map(move |c| format!("{}{}", r, c)))
    .collect();

let mut unitlist: Vec<Vec<String>> = Vec::new();
for c in cols.chars() {
    unitlist.push(rows.chars().map(|r| format!("{}{}", r, c)).collect());
}
for r in rows.chars() {
    unitlist.push(cols.chars().map(|c| format!("{}{}", r, c)).collect());
}
for rs in ["ABC", "DEF", "GHI"] {
    for cs in ["123", "456", "789"] {
        unitlist.push(rs.chars()
            .flat_map(|r| cs.chars().map(move |c| format!("{}{}", r, c)))
            .collect());
    }
}

let units: HashMap<String, Vec<Vec<String>>> = squares.iter()
    .map(|s| (s.clone(), unitlist.iter().filter(|u| u.contains(s)).cloned().collect()))
    .collect();

let peers: HashMap<String, HashSet<String>> = squares.iter()
    .map(|s| {
        let mut p: HashSet<String> = HashSet::new();
        for u in &units[s] { for sq in u { p.insert(sq.clone()); } }
        p.remove(s);
        (s.clone(), p)
    })
    .collect();

let data = SudokuData { squares, units, peers };

fn py_center(s: &str, width: usize) -> String {
    let len = s.len();
    if len >= width { return s.to_string(); }
    let marge = width - len;
    let gauche = marge / 2 + (marge & width & 1);
    let droite = marge - gauche;
    format!("{}{}{}", " ".repeat(gauche), s, " ".repeat(droite))
}

// Affiche une grille.
fn display(data: &SudokuData, values: &HashMap<String, String>) -> String {
    let width = 1 + data.squares.iter().map(|s| values[s].len()).max().unwrap();
    let dashes = "-".repeat(width * 3);
    let line = format!("{}+{}+{}", dashes, dashes, dashes);
    let mut result: Vec<String> = Vec::new();
    for r in "ABCDEFGHI".chars() {
        let mut row = String::new();
        for c in "123456789".chars() {
            let s = format!("{}{}", r, c);
            row.push_str(&py_center(&values[&s], width));
            if c == '3' || c == '6' {
                row.push('|');
            }
        }
        result.push(row);
        if r == 'C' || r == 'F' {
            result.push(line.clone());
        }
    }
    result.join("\n")
}

println!("Fonction definie : display");

Fonction definie : display


## Test du solveur

Testons le solveur sur une grille facile.

In [8]:
// Test avec une grille facile
use std::collections::{HashMap, HashSet};

#[derive(Clone)]
struct SudokuData {
    squares: Vec<String>,
    units: HashMap<String, Vec<Vec<String>>>,
    peers: HashMap<String, HashSet<String>>,
}

let digits = "123456789";
let rows = "ABCDEFGHI";
let cols = digits;

let squares: Vec<String> = rows.chars()
    .flat_map(|r| cols.chars().map(move |c| format!("{}{}", r, c)))
    .collect();

let mut unitlist: Vec<Vec<String>> = Vec::new();
for c in cols.chars() {
    unitlist.push(rows.chars().map(|r| format!("{}{}", r, c)).collect());
}
for r in rows.chars() {
    unitlist.push(cols.chars().map(|c| format!("{}{}", r, c)).collect());
}
for rs in ["ABC", "DEF", "GHI"] {
    for cs in ["123", "456", "789"] {
        unitlist.push(rs.chars()
            .flat_map(|r| cs.chars().map(move |c| format!("{}{}", r, c)))
            .collect());
    }
}

let units: HashMap<String, Vec<Vec<String>>> = squares.iter()
    .map(|s| (s.clone(), unitlist.iter().filter(|u| u.contains(s)).cloned().collect()))
    .collect();

let peers: HashMap<String, HashSet<String>> = squares.iter()
    .map(|s| {
        let mut p: HashSet<String> = HashSet::new();
        for u in &units[s] { for sq in u { p.insert(sq.clone()); } }
        p.remove(s);
        (s.clone(), p)
    })
    .collect();

let data = SudokuData { squares, units, peers };

fn assign(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    let other_values: String = values[s].chars().filter(|&c| c != d).collect();
    for d2 in other_values.chars() {
        if !eliminate(data, values, s, d2) {
            return false;
        }
    }
    true
}

fn eliminate(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    if !values[s].contains(d) {
        return true;
    }
    let new_val: String = values[s].chars().filter(|&c| c != d).collect();
    values.insert(s.to_string(), new_val);

    let s_val = values[s].clone();
    if s_val.is_empty() {
        return false;
    } else if s_val.len() == 1 {
        let d2 = s_val.chars().next().unwrap();
        for s2 in &data.peers[s] {
            if !eliminate(data, values, s2, d2) {
                return false;
            }
        }
    }

    for u in &data.units[s] {
        let dplaces: Vec<&String> = u.iter().filter(|s2| values[s2.as_str()].contains(d)).collect();
        if dplaces.is_empty() {
            return false;
        } else if dplaces.len() == 1 {
            if !assign(data, values, &dplaces[0], d) {
                return false;
            }
        }
    }
    true
}

fn grid_values(grid: &str, squares: &[String]) -> Vec<(String, char)> {
    let chars: Vec<char> = grid.chars()
        .filter(|c| c.is_ascii_digit() || *c == '0' || *c == '.')
        .collect();
    assert_eq!(chars.len(), 81);
    squares.iter().zip(chars).map(|(s, c)| (s.clone(), c)).collect()
}

fn parse_grid(data: &SudokuData, grid: &str) -> Option<HashMap<String, String>> {
    let digits = "123456789";
    let mut values: HashMap<String, String> = data.squares.iter()
        .map(|s| (s.clone(), digits.to_string()))
        .collect();
    for (s, d) in grid_values(grid, &data.squares) {
        if d != '0' && d != '.' {
            if !assign(data, &mut values, s.as_str(), d) {
                return None;
            }
        }
    }
    Some(values)
}

fn search(data: &SudokuData, values: Option<HashMap<String, String>>) -> Option<HashMap<String, String>> {
    let values = values?;
    if data.squares.iter().all(|s| values[s.as_str()].len() == 1) {
        return Some(values);
    }
    // MRV : min par (longueur, nom) en itérant sur squares (ordre lexicographique)
    let s = data.squares.iter()
        .filter(|s| values[s.as_str()].len() > 1)
        .min_by_key(|s| (values[s.as_str()].len(), s.to_string()))
        .unwrap();
    let candidates = values[s.as_str()].clone();
    for d in candidates.chars() {
        let mut new_values = values.clone();
        if assign(data, &mut new_values, s.as_str(), d) {
            if let Some(result) = search(data, Some(new_values)) {
                return Some(result);
            }
        }
    }
    None
}

fn solve(data: &SudokuData, grid: &str) -> Option<HashMap<String, String>> {
    search(data, parse_grid(data, grid))
}

fn py_center(s: &str, width: usize) -> String {
    let len = s.len();
    if len >= width { return s.to_string(); }
    let marge = width - len;
    let gauche = marge / 2 + (marge & width & 1);
    let droite = marge - gauche;
    format!("{}{}{}", " ".repeat(gauche), s, " ".repeat(droite))
}

fn display(data: &SudokuData, values: &HashMap<String, String>) -> String {
    let width = 1 + data.squares.iter().map(|s| values[s.as_str()].len()).max().unwrap();
    let dashes = "-".repeat(width * 3);
    let line = format!("{}+{}+{}", dashes, dashes, dashes);
    let mut result: Vec<String> = Vec::new();
    for r in "ABCDEFGHI".chars() {
        let mut row = String::new();
        for c in "123456789".chars() {
            let s = format!("{}{}", r, c);
            row.push_str(&py_center(&values[&s], width));
            if c == '3' || c == '6' {
                row.push('|');
            }
        }
        result.push(row);
        if r == 'C' || r == 'F' {
            result.push(line.clone());
        }
    }
    result.join("\n")
}

let easy = "..3.2.6..9..3.5..1..18.64....81.29..7.......8..67.82....26.95..8..2.3..9..5.1.3..";
println!("Puzzle:");
let values = parse_grid(&data, easy).unwrap();
println!("{}", display(&data, &values));
println!("\nResolution...");
let start = std::time::Instant::now();
let solution = solve(&data, easy);
let elapsed = start.elapsed().as_secs_f64() * 1000.0;
if let Some(sol) = solution {
    println!("\nResolu en {:.2} ms", elapsed);
    println!("{}", display(&data, &sol));
} else {
    println!("Pas de solution");
};

Puzzle:


4 8 3 |9 2 1 |6 5 7 


9 6 7 |3 4 5 |8 2 1 


2 5 1 |8 7 6 |4 9 3 


------+------+------


5 4 8 |1 3 2 |9 7 6 


... (15 outputs left out)

------+------+------


3 7 2 |6 8 9 |5 1 4 


8 1 4 |2 5 3 |7 6 9 


6 9 5 |4 1 7 |3 8 2 


### Lecture du test : la propagation au parsing résout déjà la grille

Regardez bien la sortie : la grille affichée sous « Puzzle: » ne contient **aucun point** — elle est complète. Ce n'est pas parce que l'entrée est déjà résolue : la chaîne `easy` contient bien des points (cases vides). C'est la **propagation lors du parsing** qui referme tout : chaque appel à `assign` dans `parse_grid` déclenche `eliminate` en cascade, et sur cette grille facile les indices fournis suffisent à propager jusqu'au point fixe. Le solveur `solve` re-démontre donc une grille déjà close : le temps mesuré couvre l'aller-retour complet chaîne → domaines → solution, sans qu'un seul embranchement de recherche soit nécessaire.

## Benchmark

Testons le solveur sur des puzzles de différentes difficultés.

## Exercice : Analyser l'impact de la propagation des contraintes

**Objectif**
Implémentez le solveur Norvig **sans propagation en cascade** : la fonction séparée
`solve_without_propagation` du squelette ci-dessous doit résoudre une grille
par backtracking MRV **sans jamais appeler `eliminate`** (donc sans la cascade
récursive ni la règle du singleton d'unité). Il faut écrire une **assignation
distincte** (par exemple `assign_plain`) qui fait un **forward checking minimal** :
écrire la valeur dans la case **et** retirer ce chiffre des candidats de tous les
pairs (itérativement, sans récursion). C'est ce retrait des pairs qui permet de
détecter une contradiction (domaine vide) et de déclencher le backtracking.
Comparez le nombre d'appels récursifs et le temps de résolution avec le solveur
complet `solve` sur les mêmes grilles.

**Indice**
Travaillez uniquement dans la fonction du squelette — ne modifiez pas `solve`
lui-même : le benchmark juste après cet exercice l'appelle tel quel. Votre
`assign_plain` : (1) écrit `values[s] = d.to_string()` ; (2) pour chaque pair
de `s`, retire `d` des candidats du pair — si un pair perd son dernier candidat,
retournez `false` (contradiction → backtracking). **Sans** cette étape (2), aucun
domaine ne rétrécit jamais, aucune contradiction n'est détectée, et le solveur
« résout » trivialement n'importe quelle grille en 81 assignations sans un seul
backtrack. Le backtracking MRV (choix de la case, essai des candidats) reste
identique à `search`. Pour compter les appels, un simple compteur incrémenté
en tête de la fonction de recherche suffit ; testez d'abord sur les grilles
difficiles (`Sudoku_hardest.txt`), là où la propagation en cascade fait la
plus grosse différence.

In [9]:
// EXERCICE : Analyser l'impact de la propagation des contraintes
// TODO: Implementez un solveur Norvig sans propagation de contraintes
// (uniquement assign + backtracking, pas d'eliminate)
// Comparez le nombre d'appels recursifs avec le solveur complet

use std::collections::HashMap;

fn solve_without_propagation(_grid: &str) -> Option<HashMap<String, String>> {
    None
}

println!("Exercice a completer");

Exercice a completer


In [10]:
use std::collections::{HashMap, HashSet};
use std::io::Read;

#[derive(Clone)]
struct SudokuData {
    squares: Vec<String>,
    units: HashMap<String, Vec<Vec<String>>>,
    peers: HashMap<String, HashSet<String>>,
}

let digits = "123456789";
let rows = "ABCDEFGHI";
let cols = digits;

let squares: Vec<String> = rows.chars()
    .flat_map(|r| cols.chars().map(move |c| format!("{}{}", r, c)))
    .collect();

let mut unitlist: Vec<Vec<String>> = Vec::new();
for c in cols.chars() {
    unitlist.push(rows.chars().map(|r| format!("{}{}", r, c)).collect());
}
for r in rows.chars() {
    unitlist.push(cols.chars().map(|c| format!("{}{}", r, c)).collect());
}
for rs in ["ABC", "DEF", "GHI"] {
    for cs in ["123", "456", "789"] {
        unitlist.push(rs.chars()
            .flat_map(|r| cs.chars().map(move |c| format!("{}{}", r, c)))
            .collect());
    }
}

let units: HashMap<String, Vec<Vec<String>>> = squares.iter()
    .map(|s| (s.clone(), unitlist.iter().filter(|u| u.contains(s)).cloned().collect()))
    .collect();

let peers: HashMap<String, HashSet<String>> = squares.iter()
    .map(|s| {
        let mut p: HashSet<String> = HashSet::new();
        for u in &units[s] { for sq in u { p.insert(sq.clone()); } }
        p.remove(s);
        (s.clone(), p)
    })
    .collect();

let data = SudokuData { squares, units, peers };

fn assign(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    let other_values: String = values[s].chars().filter(|&c| c != d).collect();
    for d2 in other_values.chars() {
        if !eliminate(data, values, s, d2) { return false; }
    }
    true
}

fn eliminate(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    if !values[s].contains(d) { return true; }
    let new_val: String = values[s].chars().filter(|&c| c != d).collect();
    values.insert(s.to_string(), new_val);
    let s_val = values[s].clone();
    if s_val.is_empty() { return false; }
    else if s_val.len() == 1 {
        let d2 = s_val.chars().next().unwrap();
        for s2 in &data.peers[s] {
            if !eliminate(data, values, s2, d2) { return false; }
        }
    }
    for u in &data.units[s] {
        let dplaces: Vec<&String> = u.iter().filter(|s2| values[s2.as_str()].contains(d)).collect();
        if dplaces.is_empty() { return false; }
        else if dplaces.len() == 1 {
            if !assign(data, values, &dplaces[0], d) { return false; }
        }
    }
    true
}

fn grid_values(grid: &str, squares: &[String]) -> Vec<(String, char)> {
    let chars: Vec<char> = grid.chars()
        .filter(|c| c.is_ascii_digit() || *c == '0' || *c == '.')
        .collect();
    assert_eq!(chars.len(), 81);
    squares.iter().zip(chars).map(|(s, c)| (s.clone(), c)).collect()
}

fn parse_grid(data: &SudokuData, grid: &str) -> Option<HashMap<String, String>> {
    let digits = "123456789";
    let mut values: HashMap<String, String> = data.squares.iter()
        .map(|s| (s.clone(), digits.to_string()))
        .collect();
    for (s, d) in grid_values(grid, &data.squares) {
        if d != '0' && d != '.' {
            if !assign(data, &mut values, s.as_str(), d) { return None; }
        }
    }
    Some(values)
}

fn search(data: &SudokuData, values: Option<HashMap<String, String>>) -> Option<HashMap<String, String>> {
    let values = values?;
    if data.squares.iter().all(|s| values[s.as_str()].len() == 1) {
        return Some(values);
    }
    let s = data.squares.iter()
        .filter(|s| values[s.as_str()].len() > 1)
        .min_by_key(|s| (values[s.as_str()].len(), s.to_string()))
        .unwrap();
    let candidates = values[s.as_str()].clone();
    for d in candidates.chars() {
        let mut new_values = values.clone();
        if assign(data, &mut new_values, s.as_str(), d) {
            if let Some(result) = search(data, Some(new_values)) {
                return Some(result);
            }
        }
    }
    None
}

fn solve(data: &SudokuData, grid: &str) -> Option<HashMap<String, String>> {
    search(data, parse_grid(data, grid))
}

fn load_puzzles(path: &str, max_puzzles: Option<usize>) -> std::io::Result<Vec<String>> {
    let mut file = std::fs::File::open(path)?;
    let mut buf = String::new();
    file.read_to_string(&mut buf)?;
    let mut puzzles = Vec::new();
    for line in buf.lines() {
        let line = line.trim();
        if line.len() >= 81 {
            puzzles.push(line[..81].to_string());
            if let Some(max) = max_puzzles {
                if puzzles.len() >= max { break; }
            }
        }
    }
    Ok(puzzles)
}

fn benchmark(data: &SudokuData, puzzles: &[String], name: &str) {
    println!("\nBenchmark: {} ({} puzzles)", name, puzzles.len());
    let start = std::time::Instant::now();
    let mut solved = 0;
    for puzzle in puzzles {
        if solve(data, puzzle).is_some() {
            solved += 1;
        }
    }
    let elapsed = start.elapsed().as_secs_f64() * 1000.0;
    println!("  Resolus: {}/{}", solved, puzzles.len());
    println!("  Temps total: {:.2} ms", elapsed);
    println!("  Temps moyen: {:.2} ms/puzzle", elapsed / puzzles.len() as f64);
}

// Tests
let puzzles_dir = std::env::current_dir().unwrap().join("Puzzles");
let easy = load_puzzles(puzzles_dir.join("Sudoku_Easy51.txt").to_str().unwrap(), Some(20)).unwrap();
let hard = load_puzzles(puzzles_dir.join("Sudoku_hardest.txt").to_str().unwrap(), None).unwrap();

benchmark(&data, &easy, "Puzzles Faciles");
benchmark(&data, &hard, "Puzzles Difficiles");

Benchmark: Puzzles Faciles (20 puzzles)


  Resolus: 20/20


  Temps total: 22.33 ms


  Temps moyen: 1.12 ms/puzzle


Benchmark: Puzzles Difficiles (11 puzzles)


  Resolus: 11/11


  Temps total: 21.05 ms


  Temps moyen: 1.91 ms/puzzle


### Lecture du benchmark : faciles et difficiles à un facteur proche de 1

**20/20 faciles résolus** et **11/11 difficiles** — aucun échec, aucune limite de temps. Le solveur exact termine toujours. Le **facteur entre difficile et facile** est très faible (visible dans les temps moyens affichés ci-dessus) : c'est la signature attendue d'un solveur guidé par la propagation — le nombre d'indices compte moins que la structure de leurs interactions.

Comparez avec le backtracking naïf du jumeau C# (Sudoku-07-Norvig-CSharp) : disqualifié sur les grilles difficiles là où cette version boucle en quelques millisecondes ; le solveur Norvig du même jumeau, lui, passe 10/10 grilles difficiles. Même algorithme de fond (propagation + MRV), deux ordres de grandeur d'écart avec l'absence d'inférence : avec propagation, la difficulté des grilles coûte un facteur modeste ; sans elle, elle coûte l'élimination pure et simple du solveur.

**Points clés** :

1. **Echelle absolue** : 31 grilles résolues en quelques dizaines de millisecondes cumulées — sur cette charge, le solveur n'est pas le goulot d'étranglement d'une chaîne de traitement.
2. **Où se joue la performance** : dans `assign`/`eliminate` — la propagation réduit les domaines avant que le backtracking ne soit sollicité ; l'hypothèse est que la recherche ne voit que très rarement un embranchement profond (à vérifier par instrumentation du nombre de nœuds visités, comme le propose l'exercice ci-dessus).
3. **Économie de moyens** : le solveur tient en quelques dizaines de lignes de Rust — à comparer aux solveurs spécialisés des notebooks 02 (DLX) et 10 (OR-Tools), plus rapides sur les cas extrêmes mais au prix d'une machinerie nettement plus lourde.

> **Note technique** : la clé est la combinaison propagation + MRV. La propagation résout la majeure partie de la grille dès l'assignation des indices ; le MRV fait le reste en choisissant toujours la case la plus contrainte, ce qui tend à maintenir l'arbre de recherche proche d'un chemin. Ces affirmations reposent sur les temps observés et la structure de l'algorithme ; une mesure directe du nombre de nœuds de recherche (compteur dans `search`) les confirmerait quantitativement.

## Exercice : Detection des singletons caches (Hidden Singles)

### Enonce

La technique des **singletons caches** (Hidden Singles) est déjà mise en œuvre par la seconde règle de `eliminate` : si un chiffre `d` ne peut apparaitre que dans **une seule case** d'une unité (ligne, colonne ou bloc), alors `d` y est assigne automatiquement. Cet exercice propose de **réimplémenter cette détection de façon explicite et isolée** : la fonction `hidden_singles(values)` parcourt toutes les unités et détecte ces singletons cachés sans passer par la cascade récursive de `eliminate`.

**Exemple** : Si dans la ligne A, seul `A3` peut contenir le chiffre `7` (tous les autres `A1, A2, A4...` ont elimine `7`), alors `A3 = 7` est force.

Implémentez la fonction `hidden_singles(values)` qui parcourt toutes les unités et detecte ces singletons caches de maniere explicite.

> **Indices :**

- Étape 1 : Pour chaque unité de `unitlist` et chaque chiffre `d` de 1 a 9
- Étape 2 : Collectez les cases de l'unité ou `d` est encore candidat
- Étape 3 : Si une seule case peut contenir `d`, assignez `d` a cette case avec `assign(values, s, d)`
- Étape 4 : Retournez `True` si au moins une assignation, `False` sinon

In [11]:
// EXERCICE : Detection des singletons caches (Hidden Singles)
//
// Indications :
//   - Parcourez unitlist pour chaque unite
//   - Pour chaque chiffre d (1-9), trouvez les cases de l unite ou d est candidat
//   - Si une seule case peut contenir d, assignez d a cette case
//   - Utilisez assign(values, s, d) pour chaque assignation
//   - Renvoyez True si au moins une assignation, False sinon

use std::collections::{HashMap, HashSet};

#[derive(Clone)]
struct SudokuData {
    squares: Vec<String>,
    units: HashMap<String, Vec<Vec<String>>>,
    peers: HashMap<String, HashSet<String>>,
}

let digits = "123456789";
let rows = "ABCDEFGHI";
let cols = digits;

let squares: Vec<String> = rows.chars()
    .flat_map(|r| cols.chars().map(move |c| format!("{}{}", r, c)))
    .collect();

let mut unitlist: Vec<Vec<String>> = Vec::new();
for c in cols.chars() {
    unitlist.push(rows.chars().map(|r| format!("{}{}", r, c)).collect());
}
for r in rows.chars() {
    unitlist.push(cols.chars().map(|c| format!("{}{}", r, c)).collect());
}
for rs in ["ABC", "DEF", "GHI"] {
    for cs in ["123", "456", "789"] {
        unitlist.push(rs.chars()
            .flat_map(|r| cs.chars().map(move |c| format!("{}{}", r, c)))
            .collect());
    }
}

let units: HashMap<String, Vec<Vec<String>>> = squares.iter()
    .map(|s| (s.clone(), unitlist.iter().filter(|u| u.contains(s)).cloned().collect()))
    .collect();

let peers: HashMap<String, HashSet<String>> = squares.iter()
    .map(|s| {
        let mut p: HashSet<String> = HashSet::new();
        for u in &units[s] { for sq in u { p.insert(sq.clone()); } }
        p.remove(s);
        (s.clone(), p)
    })
    .collect();

let data = SudokuData { squares, units, peers };

fn assign(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    let other_values: String = values[s].chars().filter(|&c| c != d).collect();
    for d2 in other_values.chars() {
        if !eliminate(data, values, s, d2) { return false; }
    }
    true
}

fn eliminate(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    if !values[s].contains(d) { return true; }
    let new_val: String = values[s].chars().filter(|&c| c != d).collect();
    values.insert(s.to_string(), new_val);
    let s_val = values[s].clone();
    if s_val.is_empty() { return false; }
    else if s_val.len() == 1 {
        let d2 = s_val.chars().next().unwrap();
        for s2 in &data.peers[s] {
            if !eliminate(data, values, s2, d2) { return false; }
        }
    }
    for u in &data.units[s] {
        let dplaces: Vec<&String> = u.iter().filter(|s2| values[s2.as_str()].contains(d)).collect();
        if dplaces.is_empty() { return false; }
        else if dplaces.len() == 1 {
            if !assign(data, values, &dplaces[0], d) { return false; }
        }
    }
    true
}

fn grid_values(grid: &str, squares: &[String]) -> Vec<(String, char)> {
    let chars: Vec<char> = grid.chars()
        .filter(|c| c.is_ascii_digit() || *c == '0' || *c == '.')
        .collect();
    assert_eq!(chars.len(), 81);
    squares.iter().zip(chars).map(|(s, c)| (s.clone(), c)).collect()
}

fn parse_grid(data: &SudokuData, grid: &str) -> Option<HashMap<String, String>> {
    let digits = "123456789";
    let mut values: HashMap<String, String> = data.squares.iter()
        .map(|s| (s.clone(), digits.to_string()))
        .collect();
    for (s, d) in grid_values(grid, &data.squares) {
        if d != '0' && d != '.' {
            if !assign(data, &mut values, s.as_str(), d) { return None; }
        }
    }
    Some(values)
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn hidden_singles(_values: &mut HashMap<String, String>) -> bool {
    false
}

let test_grid_hs = "..3.2.6..9..3.5..1..18.64....81.29..7.......8..67.82....26.95..8..2.3..9..5.1.3..";
let values_hs = parse_grid(&data, test_grid_hs);
if let Some(mut vals) = values_hs {
    let result = hidden_singles(&mut vals);
    println!("Singletons caches detectes : {}", py_bool(result));
} else {
    println!("Grille invalide");
};

Singletons caches detectes : False


## Exercice : Porter l'algorithme Norvig sur un mini-Sudoku 4x4

### Enonce

Le Sudoku 4x4 est une version simplifiee : une grille 4x4 divisee en 4 blocs 2x2, avec des chiffres de 1 a 4. Adaptez l'algorithme de Norvig (propagation de contraintes + backtracking MRV) pour résoudre ce mini-Sudoku.

Grille de test (0 = case vide) :
```
1 0 | 0 4
0 4 | 1 0
-----
0 1 | 4 0
4 0 | 0 1
```

> **Indices :**

- Remplacez `digits = '123456789'` par `digits = '1234'`
- Remplacez `rows = 'ABCDEFGHI'` par `rows = 'ABCD'`
- Redefinissez les unités pour les blocs 2x2 (non 3x3)
- Les fonctions `assign`, `eliminate` et `search` n'ont pas besoin d'être modifiees

In [12]:
// EXERCICE : Mini-Sudoku 4x4 avec l'algorithme de Norvig
use std::collections::{HashMap, HashSet};

#[derive(Clone)]
struct Sudoku4Data {
    squares: Vec<String>,
    units: HashMap<String, Vec<Vec<String>>>,
    peers: HashMap<String, HashSet<String>>,
}

let digits4 = "1234";
let rows4 = "ABCD";
let cols4 = digits4;

let squares4: Vec<String> = rows4.chars()
    .flat_map(|r| cols4.chars().map(move |c| format!("{}{}", r, c)))
    .collect();

let mut unitlist4: Vec<Vec<String>> = Vec::new();
for r in rows4.chars() {
    unitlist4.push(cols4.chars().map(|c| format!("{}{}", r, c)).collect());
}
for c in cols4.chars() {
    unitlist4.push(rows4.chars().map(|r| format!("{}{}", r, c)).collect());
}
for rs in ["AB", "CD"] {
    for cs in ["12", "34"] {
        unitlist4.push(rs.chars()
            .flat_map(|r| cs.chars().map(move |c| format!("{}{}", r, c)))
            .collect());
    }
}

let units4: HashMap<String, Vec<Vec<String>>> = squares4.iter()
    .map(|s| (s.clone(), unitlist4.iter().filter(|u| u.contains(s)).cloned().collect()))
    .collect();

let peers4: HashMap<String, HashSet<String>> = squares4.iter()
    .map(|s| {
        let mut p: HashSet<String> = HashSet::new();
        for u in &units4[s] { for sq in u { p.insert(sq.clone()); } }
        p.remove(s);
        (s.clone(), p)
    })
    .collect();

let data4 = Sudoku4Data { squares: squares4, units: units4, peers: peers4 };

fn assign4(data: &Sudoku4Data, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    let other_values: String = values[s].chars().filter(|&c| c != d).collect();
    for d2 in other_values.chars() {
        if !eliminate4(data, values, s, d2) { return false; }
    }
    true
}

fn eliminate4(data: &Sudoku4Data, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    if !values[s].contains(d) { return true; }
    let new_val: String = values[s].chars().filter(|&c| c != d).collect();
    values.insert(s.to_string(), new_val);
    let s_val = values[s].clone();
    if s_val.is_empty() { return false; }
    else if s_val.len() == 1 {
        let d2 = s_val.chars().next().unwrap();
        for s2 in &data.peers[s] {
            if !eliminate4(data, values, s2, d2) { return false; }
        }
    }
    for u in &data.units[s] {
        let dplaces: Vec<&String> = u.iter().filter(|s2| values[s2.as_str()].contains(d)).collect();
        if dplaces.is_empty() { return false; }
        else if dplaces.len() == 1 {
            if !assign4(data, values, &dplaces[0], d) { return false; }
        }
    }
    true
}

fn parse_grid4(data: &Sudoku4Data, grid: &str) -> Option<HashMap<String, String>> {
    let digits4 = "1234";
    let mut values: HashMap<String, String> = data.squares.iter()
        .map(|s| (s.clone(), digits4.to_string()))
        .collect();
    for (s, d) in data.squares.iter().zip(grid.chars()) {
        if digits4.contains(d) {
            if !assign4(data, &mut values, s, d) { return None; }
        }
    }
    Some(values)
}

fn search4(data: &Sudoku4Data, values: Option<HashMap<String, String>>) -> Option<HashMap<String, String>> {
    let values = values?;
    if data.squares.iter().all(|s| values[s.as_str()].len() == 1) {
        return Some(values);
    }
    let s = data.squares.iter()
        .filter(|s| values[s.as_str()].len() > 1)
        .min_by_key(|s| (values[s.as_str()].len(), s.to_string()))
        .unwrap();
    let candidates = values[s.as_str()].clone();
    for d in candidates.chars() {
        let mut new_values = values.clone();
        if assign4(data, &mut new_values, s.as_str(), d) {
            if let Some(result) = search4(data, Some(new_values)) {
                return Some(result);
            }
        }
    }
    None
}

fn solve_4x4(data: &Sudoku4Data, grid: &str) -> Option<HashMap<String, String>> {
    search4(data, parse_grid4(data, grid))
}

// Stub à compléter : display_4x4
// Indications :
//   - Si values est None, renvoyer "Pas de solution"
//   - Calculer width = 1 + max(longueur des candidats sur les 16 cases)
//   - Redéfinir un centrage local dans cette cellule (marge/2 à gauche, reste à droite)
//   - Construire chaque ligne : 4 cases centrées, séparées par un espace
//   - Insérer un séparateur "|" après la 2e colonne
//   - Insérer une ligne de tirets ("----") après la 2e rangée
//   - Joindre les lignes avec "\n"

fn display_4x4(_values: &Option<HashMap<String, String>>) -> String {
    "Exercice a completer".to_string()
}

let test_4x4 = "1004041001404001";
let solution_4x4 = solve_4x4(&data4, test_4x4);
println!("Puzzle 4x4:");
println!("{}", display_4x4(&parse_grid4(&data4, test_4x4)));
println!("\nSolution 4x4:");
println!("{}", display_4x4(&solution_4x4));

Puzzle 4x4:


Exercice a completer


Solution 4x4:


Exercice a completer


## Exercice : Detection de paires nues (Naked Pairs)

### Enonce

La technique des **paires nues** (Naked Pairs) est une heuristique de propagation supplementaire. Si deux cases d'une même unité ont exactement les mêmes deux candidats (par exemple `{2, 5}`), alors ces deux valeurs peuvent être eliminees de toutes les autres cases de cette unité.

**Exemple** : Si `A1 = "25"` et `A4 = "25"` dans la ligne A, alors `2` et `5` peuvent être supprimes des candidats de `A2, A3, A5, A6, A7, A8, A9`.

Implantez une fonction `naked_pairs(values)` qui :
1. Parcourt chaque unité (lignes, colonnes, blocs)
2. Detecte les paires de cases partageant exactement 2 candidats identiques
3. Elimine ces candidats des autres cases de l'unité
4. Renvoie `True` si au moins une elimination a eu lieu, `False` sinon

> **Indices :**

- Pour chaque unité, collectez les cases ayant exactement 2 candidats
- Comparez les ensembles de candidats deux a deux
- Utilisez la fonction `eliminate` existante pour propager les suppressions
- Pensez a vérifier que la case dont on elimine n'est pas l'une des deux cases de la paire

In [13]:
// EXERCICE : Detection de paires nues (Naked Pairs)
// TODO: Implantez la fonction naked_pairs(values) decrite ci-dessus
//
// Indications :
//   - Parcourez unitlist pour chaque unite
//   - Pour chaque unite, trouvez les cases avec exactement 2 candidats
//   - Si deux cases ont les memes 2 candidats, eliminez-les des autres cases
//   - Utilisez eliminate(values, s, d) pour chaque elimination
//   - Renvoyez True si au moins une elimination, False sinon

use std::collections::{HashMap, HashSet};

#[derive(Clone)]
struct SudokuData {
    squares: Vec<String>,
    units: HashMap<String, Vec<Vec<String>>>,
    peers: HashMap<String, HashSet<String>>,
}

let digits = "123456789";
let rows = "ABCDEFGHI";
let cols = digits;

let squares: Vec<String> = rows.chars()
    .flat_map(|r| cols.chars().map(move |c| format!("{}{}", r, c)))
    .collect();

let mut unitlist: Vec<Vec<String>> = Vec::new();
for c in cols.chars() {
    unitlist.push(rows.chars().map(|r| format!("{}{}", r, c)).collect());
}
for r in rows.chars() {
    unitlist.push(cols.chars().map(|c| format!("{}{}", r, c)).collect());
}
for rs in ["ABC", "DEF", "GHI"] {
    for cs in ["123", "456", "789"] {
        unitlist.push(rs.chars()
            .flat_map(|r| cs.chars().map(move |c| format!("{}{}", r, c)))
            .collect());
    }
}

let units: HashMap<String, Vec<Vec<String>>> = squares.iter()
    .map(|s| (s.clone(), unitlist.iter().filter(|u| u.contains(s)).cloned().collect()))
    .collect();

let peers: HashMap<String, HashSet<String>> = squares.iter()
    .map(|s| {
        let mut p: HashSet<String> = HashSet::new();
        for u in &units[s] { for sq in u { p.insert(sq.clone()); } }
        p.remove(s);
        (s.clone(), p)
    })
    .collect();

let data = SudokuData { squares, units, peers };

fn assign(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    let other_values: String = values[s].chars().filter(|&c| c != d).collect();
    for d2 in other_values.chars() {
        if !eliminate(data, values, s, d2) { return false; }
    }
    true
}

fn eliminate(data: &SudokuData, values: &mut HashMap<String, String>, s: &str, d: char) -> bool {
    if !values[s].contains(d) { return true; }
    let new_val: String = values[s].chars().filter(|&c| c != d).collect();
    values.insert(s.to_string(), new_val);
    let s_val = values[s].clone();
    if s_val.is_empty() { return false; }
    else if s_val.len() == 1 {
        let d2 = s_val.chars().next().unwrap();
        for s2 in &data.peers[s] {
            if !eliminate(data, values, s2, d2) { return false; }
        }
    }
    for u in &data.units[s] {
        let dplaces: Vec<&String> = u.iter().filter(|s2| values[s2.as_str()].contains(d)).collect();
        if dplaces.is_empty() { return false; }
        else if dplaces.len() == 1 {
            if !assign(data, values, &dplaces[0], d) { return false; }
        }
    }
    true
}

fn grid_values(grid: &str, squares: &[String]) -> Vec<(String, char)> {
    let chars: Vec<char> = grid.chars()
        .filter(|c| c.is_ascii_digit() || *c == '0' || *c == '.')
        .collect();
    assert_eq!(chars.len(), 81);
    squares.iter().zip(chars).map(|(s, c)| (s.clone(), c)).collect()
}

fn parse_grid(data: &SudokuData, grid: &str) -> Option<HashMap<String, String>> {
    let digits = "123456789";
    let mut values: HashMap<String, String> = data.squares.iter()
        .map(|s| (s.clone(), digits.to_string()))
        .collect();
    for (s, d) in grid_values(grid, &data.squares) {
        if d != '0' && d != '.' {
            if !assign(data, &mut values, s.as_str(), d) { return None; }
        }
    }
    Some(values)
}

fn naked_pairs(_values: &mut HashMap<String, String>) -> bool {
    false
}

let test_naked = "4.....8.5.3..........7......2.....6.....8.4......1.......6.3.7.5..2.....1.4......";
let values_test = parse_grid(&data, test_naked).unwrap();
println!("Candidats avant naked_pairs :");
println!("  A2 = {}, A8 = {}", values_test["A2"], values_test["A8"]);
// result = naked_pairs(values_test)
// print(f"Eliminations effectuees : {result}")
// print(f"  A2 = {values_test['A2']}, A8 = {values_test['A8']}")

Candidats avant naked_pairs :


  A2 = 1679, A8 = 1239


### Lecture des candidats : la notation compacte de Norvig

`A2 = 1679` se lit : la cellule A2 admet encore **quatre candidats — 1, 6, 7, 9** — écrits en concaténation, exactement comme dans l'essai d'origine de Norvig dont ce notebook est le portage. Cette notation n'est pas qu'une économie d'affichage : c'est le domaine lui-même, la valeur de travail de l'algorithme (`1679` est la chaîne que `eliminate` rogne caractère par caractère). L'exercice de détection de paires nues qui suit exploite ces listes : si deux cellules d'une même unité ont exactement le même couple de candidats, ces deux valeurs peuvent être éliminées de tous leurs autres pairs.

## Synthèse

Ce notebook a présenté l'approche élégante de Peter Norvig pour résoudre le Sudoku avec une implémentation compacte. La méthode repose sur deux piliers : la **propagation de contraintes** (élimination des candidats impossibles via les pairs et les unités) et le **backtracking** guidé par l'heuristique MRV (choisir la case avec le moins de valeurs possibles). Le benchmark montre des performances remarquables (quelques millisecondes par puzzle en moyenne), avec une faible sensibilité à la difficulté de la grille. Cette implémentation illustre un principe fondamental en intelligence artificielle : un bon encodage du problème (représentation par dictionnaire de candidats) combiné à une heuristique pertinente (MRV) peut surpasser des approches algorithmiquement plus complexes. Les exercices proposés (mini-Sudoku 4x4, paires nues) permettent d'étendre le solveur avec des techniques de propagation avancées.

## Synthèse

### Points clés du solveur Norvig

1. **Propagation** : Réduit les domaines avant la recherche
2. **MRV** : Choix intelligent de la case suivante
3. **Backtracking simple** : Pas de besoin d'algorithmes complexes

### Performances mesurées (ce notebook)

| Type | Résultat | Taux de succès |
|------|----------|----------------|
| Faciles (20 puzzles) | voir sortie ci-dessus | 100% (20/20) |
| Difficiles (11 puzzles) | voir sortie ci-dessus | 100% (11/11) |

### Références

- [Peter Norvig's Sudoku Solver](https://norvig.com/sudoku.html)
- [Constraint Satisfaction](https://en.wikipedia.org/wiki/Constraint_satisfaction_problem)

***

***

**Retour au sommaire** : [Index Sudoku](README.md)

## À retenir

L'approche de **Peter Norvig** (2006) — **~80 instructions dans son article** — résout le Sudoku grâce à une **propagation de contraintes** élégante : chaque case porte l'ensemble de ses valeurs possibles, et deux règles (`assign` / `eliminate`) éliminent en cascade les valeurs impossibles. Beaucoup de puzzles se résolvent *sans aucun backtracking*.

- La structure du problème se résume en trois nombres : **81 cases**, **27 unités** (9 rangées + 9 colonnes + 9 blocs) et **20 paires** pour la case A1 (ses voisins).
- Le puzzle facile du test est résolu en quelques millisecondes ; sur 20 puzzles faciles, **20/20** ; sur 11 puzzles difficiles, **11/11** — le facteur entre les deux est très faible, signe que la propagation fait l'essentiel du travail.
- L'apport de Norvig est **pédagogique** : montrer qu'une propagation bien conçue fait l'essentiel du travail, et que le backtracking n'intervient qu'en dernier recours.

## Vérifiez votre compréhension

**1. Combien d'« unités » compte le Sudoku, et que sont-elles ?**

<details><summary>Réponse</summary>

**27 unités** : les 9 rangées, les 9 colonnes et les 9 blocs 3×3. Chaque case appartient à exactement 3 unités (une ligne, une colonne, un bloc), et c'est sur ces unités que la propagation élimine les valeurs — une case ne peut partager une valeur avec aucun de ses **20 paires** pour A1.

</details>

**2. Le solveur de Norvig a-t-il besoin de backtracking sur les puzzles du benchmark ?**

<details><summary>Réponse</summary>

Pratiquement non : les 20 puzzles faciles et les 11 difficiles sont tous résolus en quelques millisecondes — des temps si bas que la propagation de contraintes (`eliminate` en cascade) suffit dans la quasi-totalité des cas. Le backtracking reste dans le code comme **filet de sécurité** pour les rares puzzles que la propagation seule ne clôt pas.

</details>

**3. En quoi l'approche de Norvig diffère-t-elle de la formulation CSP du carnet précédent ?**

<details><summary>Réponse</summary>

Même idée (propagation de contraintes), autre style : Norvig n'instancie pas un algorithme CSP générique (MRV, forward checking, MAC) mais écrit la propagation « à la main » en ~80 instructions **dans son article**, où `assign` et `eliminate` se appellent mutuellement. C'est plus **lisible** que complet — le carnet [Sudoku-06](Sudoku-06-AIMA-CSP-Python.ipynb) montre le spectre d'algorithmes qu'on gagne avec la formulation CSP formelle.

</details>

## Pour aller plus loin

- **[Sudoku-08 — Stratégies Humaines](Sudoku-08-HumanStrategies-Python.ipynb)** : le carnet suivant — formaliser les « tricks » que les humains utilisent (candidate unique, hidden single…) en algorithmes.

- [Sudoku-07 — Norvig (C#)](Sudoku-07-Norvig-CSharp.ipynb) : le jumeau C# de la propagation de Norvig.

### Lectures

- Norvig, *Solving Every Sudoku Puzzle* (2006) — l'article originel de l'approche en ~80 instructions.

> Carnet issu de l'EPIC #26510.